# Week 3, day 3 (afternoon) — Worksheet 01 SOLUTIONS: Snowsight and the sample data

The five lab tasks, plus the orientation queries around them.

**These solutions were not executed.** There is no Snowflake connection in this
lab environment, so the SQL is written from the lab material and the row counts
quoted are the lab's own — verified by you when you run them, not by me. Where a
number is genuinely fixed by the TPC-H specification rather than by a claim on a
page, the answer says so.

That distinction matters more than it sounds. Worksheets 03 to 05 run locally and
every figure in them is observed output. These two do not. Knowing which of your
sources has actually been run is the same skill the rest of the day is about.

PART A — where you are

### Question 1

Open a new worksheet and run `SELECT CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_ROLE();`. Record all four values. If the warehouse is null, select one in the context selector and run it again.
> **NOTE:** three of these four can be null and the query still returns a row. Only the warehouse being null stops other queries from running.

```sql
SELECT CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_ROLE();
```

Four values that together are your execution context. On a fresh trial account
the role is usually `ACCOUNTADMIN` and the database and schema are null.

The asymmetry in the note is the practical part. A null database or schema is
harmless as long as you fully qualify every table name — `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER`
names the table completely and needs no context at all. A null **warehouse** is
different: the warehouse is the compute, and without one there is nothing to run
the query on. The error is *"No active warehouse selected in the current
session"*, and it is the single most common thing to hit in a fresh worksheet.

Worth internalising because it is unlike most databases you have used: in
Snowflake, storage and compute are separate things that you select separately.
The data is always there. Whether anything can read it depends on the warehouse.

### Question 2

Confirm the sample data is there: run `SHOW SCHEMAS IN DATABASE SNOWFLAKE_SAMPLE_DATA;` and record the schema names. Then run `SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER;` and `... .ORDERS;`.

```sql
SHOW SCHEMAS IN DATABASE SNOWFLAKE_SAMPLE_DATA;

SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER;
SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;
```

`SNOWFLAKE_SAMPLE_DATA` holds several TPC-H schemas at different scales —
`TPCH_SF1`, `TPCH_SF10`, `TPCH_SF100`, `TPCH_SF1000` — plus `TPCDS_*` and
`INFORMATION_SCHEMA`. The `SF` number is the **scale factor**: `SF1` is roughly
1 GB, `SF1000` roughly 1 TB of the same schema. Same tables, same columns,
different row counts — which makes it the standard place to see how a query
behaves as data grows.

The two counts are **150,000 customers and 1,500,000 orders**, and these are not
a claim on a page: TPC-H defines `CUSTOMER` as 150,000 x SF rows and `ORDERS` as
1,500,000 x SF. At SF1 that is exactly those figures, in every Snowflake account,
for anyone. If yours differ, you are querying a different schema.

Ten orders per customer, on average. Keep that ratio in mind for Q7 — it is why
joining these two tables produces 1.5 million rows and not 150,000.

### Question 3

Look at the shape of one table before querying it: `SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER LIMIT 5;`. Record the column names, and note which column you would join to `ORDERS` on.
> **NOTE:** `LIMIT 5` on a 150,000-row table and `SELECT *` with no limit cost the same to write and very different amounts to run. Get into the habit now.

```sql
SELECT * FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER LIMIT 5;
```

Eight columns: `C_CUSTKEY`, `C_NAME`, `C_ADDRESS`, `C_NATIONKEY`, `C_PHONE`,
`C_ACCTBAL`, `C_MKTSEGMENT`, `C_COMMENT`.

The join column is **`C_CUSTKEY`**, which matches `O_CUSTKEY` in `ORDERS`. TPC-H
prefixes every column with a letter for its table — `C_` customer, `O_` orders,
`N_` nation, `R_` region, `L_` lineitem, `P_` part, `S_` supplier — so you can
always tell where a column came from in a join, which is a convention worth
stealing.

`C_NATIONKEY` is the other foreign key, into `NATION`. That is the chain Q6 uses:
customer → nation → region.

On the `LIMIT`: Snowflake charges for compute time, so a habit that costs nothing
in a 5-row toy table costs real money at SF1000. `SELECT *` with no limit on a
1 TB table is a genuine mistake, not a style preference. Look at five rows first,
always.

PART B — the five lab tasks

### Question 4

**Task 1 — customers by market segment.** Using `CUSTOMER`, return the number of customers in each market segment (`C_MKTSEGMENT`), largest segment first.

```sql
SELECT
    C_MKTSEGMENT,
    COUNT(*) AS CUSTOMER_COUNT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
GROUP BY C_MKTSEGMENT
ORDER BY CUSTOMER_COUNT DESC;
```

Five rows — `AUTOMOBILE`, `BUILDING`, `FURNITURE`, `HOUSEHOLD`, `MACHINERY` —
each holding roughly 30,000 of the 150,000 customers.

The distribution is close to even, which is the interesting part of the answer
rather than an aside. TPC-H assigns segments at random, so the counts differ by a
few hundred at most. That means the `ORDER BY` genuinely determines what appears
at the top, and re-running it against `TPCH_SF10` would very likely give a
different ranking of the same five segments.

Which is the general caution: a ranking over near-equal groups is not a finding.
"`BUILDING` is our largest segment" is a true statement about a 0.3% gap, and it
will reverse. Look at the spread before you report the order.

### Question 5

**Task 2 — orders by order priority.** Using `ORDERS`, return the number of orders for each `O_ORDERPRIORITY` value, most orders first.

```sql
SELECT
    O_ORDERPRIORITY,
    COUNT(*) AS ORDER_COUNT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
GROUP BY O_ORDERPRIORITY
ORDER BY ORDER_COUNT DESC;
```

Five rows — `1-URGENT`, `2-HIGH`, `3-MEDIUM`, `4-NOT SPECIFIED`, `5-LOW` — each
near 300,000 of the 1,500,000 orders. The same near-even split as Q4, so the same
caution applies.

The values themselves are worth a second look. They are numbered strings, not
plain labels, and the number is there so `ORDER BY O_ORDERPRIORITY` sorts them
into meaningful order — urgent first — instead of alphabetically, which would
give `1-URGENT, 2-HIGH, 3-MEDIUM, 4-NOT SPECIFIED, 5-LOW` only by luck of the
digit. Without the prefix, alphabetical order is `HIGH, LOW, MEDIUM, NOT
SPECIFIED, URGENT`: meaningless.

Compare with the `PRIORITY` column in Worksheet 03 — `"Critical"`, `"High"`,
`"Low"`, `"Medium"`, `"Not Specified"`, unnumbered and wrapped in quotes. Same
concept, one designed and one not. Ordering that column requires a `CASE`
expression somebody has to write and maintain.

### Question 6

**Task 3 — nations in a region.** Using `NATION` and `REGION`, return the names of all nations (`N_NAME`) in the region named `ASIA`. Join on `N_REGIONKEY = R_REGIONKEY`.
> **NOTE:** `R_NAME` is padded in TPC-H — the column is `CHAR(25)`. If `= 'ASIA'` returns nothing, that is why; try `TRIM(r.R_NAME) = 'ASIA'`.

```sql
SELECT
    n.N_NAME
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION n
JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION r
    ON n.N_REGIONKEY = r.R_REGIONKEY
WHERE r.R_NAME = 'ASIA'
ORDER BY n.N_NAME;
```

Five rows: `CHINA`, `INDIA`, `INDONESIA`, `JAPAN`, `VIETNAM`.

Small tables — 25 nations, 5 regions — so this is about the join, not the volume.
Note the shape: filter on a column from one table (`r.R_NAME`), select a column
from the other (`n.N_NAME`). The region name never appears in the output; the
join exists only to make the filter possible. That is the most common reason to
join a dimension table.

**If it returns nothing**, the cause is padding. TPC-H declares `R_NAME` as
`CHAR(25)`, and `CHAR` is blank-padded to its full width, so the stored value may
be `'ASIA'` followed by 21 spaces. Snowflake usually ignores trailing blanks when
comparing, but it is not universal across systems, and `TRIM(r.R_NAME) = 'ASIA'`
is the reliable form.

Add it to the list from Worksheet 03: values that do not equal what they display
as. There it was quote characters, here it is trailing spaces. Both produce an
empty result set rather than an error, and an empty result set reads exactly like
a real "none found".

### Question 7

**Task 4 — total spend by market segment.** Using `CUSTOMER` and `ORDERS`, return the total `O_TOTALPRICE` for each `C_MKTSEGMENT`, rounded to whole numbers, highest total first. Join on the customer key.

```sql
SELECT
    c.C_MKTSEGMENT,
    ROUND(SUM(o.O_TOTALPRICE)) AS TOTAL_SPEND
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER c
JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS o
    ON c.C_CUSTKEY = o.O_CUSTKEY
GROUP BY c.C_MKTSEGMENT
ORDER BY TOTAL_SPEND DESC;
```

Five rows, one per segment, with totals in the hundreds of billions — because
this sums 1.5 million order values, not 150,000.

That is the number to hold on to. The join reads 150,000 customers and 1,500,000
orders and produces **1,500,000 rows**, one per order, each carrying its
customer's segment. Every customer row is repeated about ten times.

Here that is correct, and it is correct for a specific reason: `C_CUSTKEY` is
unique in `CUSTOMER`, so each order matches exactly one customer and the fan-out
is entirely on the orders side, which is the side being summed. Each order's
value is counted once.

Compare Worksheet 05 Q9, where the same query shape inflated a total by 606,705.92
— because there the key was *not* unique on the dimension side, so fact rows were
duplicated. Same SQL, opposite outcome, and the only difference is a property of
the dimension table nobody checked.

The check is one query, and it is the one to run before any join to a table you
did not build:

```sql
SELECT COUNT(*), COUNT(DISTINCT C_CUSTKEY)
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER;
```

Equal means safe.

### Question 8

**Task 5 — highest value orders for one segment.** Using `CUSTOMER` and `ORDERS`, return the ten highest value orders placed by customers in the `AUTOMOBILE` segment: customer name, order date, order total, largest first.

```sql
SELECT
    c.C_NAME,
    o.O_ORDERDATE,
    o.O_TOTALPRICE
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER c
JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS o
    ON c.C_CUSTKEY = o.O_CUSTKEY
WHERE c.C_MKTSEGMENT = 'AUTOMOBILE'
ORDER BY o.O_TOTALPRICE DESC
LIMIT 10;
```

Ten rows, largest order total first, with the top values above 500,000.

Read the clause order, because it is the whole lesson: **join**, then `WHERE`
narrows 1.5 million rows to roughly a fifth of them, then `ORDER BY` sorts what
survives, then `LIMIT` keeps ten. Filter before sort — sorting 1.5 million rows to
throw away all but ten is work you do not need to do, and while Snowflake's
optimiser will usually rearrange this for you, writing it in the right order is
free.

One thing this result cannot tell you: whether any customer appears twice.
`LIMIT 10` over orders returns the ten largest **orders**, and one customer with
three enormous orders occupies three of those rows. If the question you actually
have is "who are our biggest automotive customers", this query does not answer it
— that needs `SUM(o.O_TOTALPRICE) ... GROUP BY c.C_NAME`.

Worth being precise about, because both queries produce a plausible top-ten list
with a customer name and a large number beside it, and they answer different
questions.

PART C — what the query actually did

### Question 9

Open the **Query Profile** for your Task 4 query (Query History → click the query → Profile). Record the total execution time, the number of partitions scanned out of total, and the percentage of time in the largest node.
> **NOTE:** this is the tab that answers "why was that slow". Learn where it is before you need it.

The Query Profile is in **Query History → click the query → Profile**, and it
is the answer to "why was that slow".

What to read, in the order it matters:

**Partitions scanned / partitions total.** Snowflake stores tables in immutable
micro-partitions and skips any it can prove are irrelevant. Scanning 5 of 200 is
excellent pruning; scanning 200 of 200 means the filter did not help and the
query read the whole table. This is usually the single most useful number on the
page.

**Percentage of execution time by node**, in the tree on the right. One node
holding most of the time tells you where to look. A `TableScan` dominating means
the query is I/O-bound and pruning is the lever; a `Join` dominating usually
means row explosion, which is Q7's fan-out showing up as a performance symptom
rather than a wrong number.

**Bytes spilled to local/remote storage.** Anything above zero means the
warehouse ran out of memory and started writing to disk. Remote spilling in
particular is a large slowdown and the usual signal to size up.

Your Task 4 query joins 150,000 to 1,500,000 rows with no `WHERE` clause, so
expect a full scan of both tables and most of the time in the join and aggregate
nodes. There is nothing to prune — the query genuinely needs every row.

The habit worth forming: look at the profile of a query that ran *fine*, once,
now, while there is no pressure. You will read it much faster later when
something is on fire.

### Question 10

Run `SELECT COUNT(*) FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;` twice in a row and record both execution times from Query History. Explain the difference. Then suspend your warehouse with `ALTER WAREHOUSE <your_warehouse> SUSPEND;`.
> **NOTE:** do not skip the suspend. A warehouse left running bills for idle time.

The second run returns almost instantly — typically tens of milliseconds against
a first run of a second or more.

That is the **result cache**. Snowflake keeps query results for 24 hours and
serves a byte-identical query from cache when the underlying data has not
changed. It consumes **no warehouse credits at all**; it will even answer while
the warehouse is suspended.

Two consequences worth knowing.

Benchmarking is the obvious trap. A second run measures the cache, not the query,
and the way to defeat it is `ALTER SESSION SET USE_CACHED_RESULT = FALSE;`. Any
"we made it 40x faster" that came from running the same query twice is measuring
nothing.

The other consequence is the more useful one: the cache is invalidated by changes
to the underlying data, so a fast repeat is also weak evidence that nothing has
been written to the table since. Not a check you would rely on, but it explains
why a dashboard sometimes goes slow for no visible reason — someone loaded data.

Then suspend:

```sql
ALTER WAREHOUSE <your_warehouse> SUSPEND;
```

Snowflake bills warehouse time by the second with a **60-second minimum** per
resume, and an idle running warehouse bills for being idle. Most have
`AUTO_SUSPEND` set to a few minutes, so this is belt-and-braces — but on a trial
account with limited credits, a warehouse left running overnight is how people
lose them. Check with:

```sql
SHOW WAREHOUSES;
```

The `state` column should read `SUSPENDED`.